## PingOne Overview

PingOne is a cloud-based identity and access management service that provides secure identity solutions for enterprises, enabling seamless authentication and authorization across applications and services.

Key Features:
* **Single Sign-On (SSO)** - Users authenticate once to access multiple applications
* **Multi-Factor Authentication (MFA)** - Enhanced security through additional verification methods
* **Adaptive Authentication** - Risk-based authentication policies based on user behavior and context
* **Universal Directory** - Centralized user management and profile synchronization
* **API Access Management** - OAuth 2.0 and OpenID Connect support for API security

## Learning Objective

PingOne can be used as an identity provider on AgentCore Identity to authenticate users and have them authorize the agent to access protected resources on their behalf. In this notebook we will explore the use of PingOne for inbound authentication - authenticating users before they can invoke an agent.

## Tutorial Architecture

Users authenticate with PingOne using the device authorization flow, then use the resulting JWT to invoke agents on AgentCore Runtime.
<figure style="width: 70%;">
    <img src="/images/inbound_auth_diagram.png" style="width: 70%; height: auto;" alt="Detailed AWS Cloud Architecture">
</figure>

## Prerequisites

- PingOne environment
- PingOne IAM rights to create new IAM resources, applications, and users
- Python 3.10+
- AWS Credentials
- AWS IAM rights to create a new AgentCore Agent
- Amazon Bedrock AgentCore SDK
- Strands Agents
- AWS region that supports Bedrock AgentCore. Refer https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/agentcore-regions.html

## Step 1: Setup PingOne for use with AgentCore

### 1.1: Create a New Resource

**This will represent AgentCore Runtime, defining the audience and scope that valid tokens must include for access.**

1. Login to your PingOne environment
2. Select **Applications**, then **Resources**, and add a resource
3. Configure the resource:
    - For **Resource Name** and **Audience**, enter **agentcore_runtime**
    - For **Scopes**, add **agent:invoke**

<figure>
    <img src="/images/inbound_resource_config.png">
</figure>

### 1.2: Create a New Application

**This will represent the client application, configured to request tokens with the required audience and scope for AgentCore Runtime access.**

1. Select **Applications**, then **Applications**, and add an application
2. Select **OIDC Web App** as the **Application Type**
3. Configure the application:
    - For **Application Name**, enter **Amazon Bedrock Chat App**
    - For **Token Auth Method**, select **Client Secret Basic**
    - For **Grant Type**, select **Authorization Code** and **Device Authorization**
        - **Note:** While Authorization Code is the standard for web apps, we use the Device Authorization grant in this notebook to allow for seamless user authentication within the Python environment without requiring a local redirect listener.
    - For **Redirect URIs**, use "https://bedrock-agentcore.your-aws-region.amazonaws.com/identities/oauth2/callback"
    - For **Resources**, add the **agent:invoke** scope on the **agentcore_runtime** resource configured in step 1.1

<figure>
    <img src="/images/inbound_application_config.png">
</figure>

### 1.3: Create a Test User

**This user will authenticate via the device authorization flow to test the protected agent endpoint.**

1. Select **Directory**, then **Users** and add a user
2. Fill in the details of the test user that will be used for this demo

<figure>
    <img src="/images/inbound_user_config.png">
</figure>

## Step 2: Notebook Setup and AWS Configuration

### 2.1: Install Dependencies for Notebook

In [ ]:
# Install required packages from requirements.txt.
%pip install --force-reinstall -U -r ../requirements.txt --quiet

### 2.2: Load Environment Variables for Notebook

In [ ]:
from dotenv import load_dotenv
import os
load_dotenv()

PINGONE_CLIENT_ID = os.environ['INBOUND_PINGONE_CLIENT_ID']
PINGONE_CLIENT_SECRET = os.environ['INBOUND_PINGONE_CLIENT_SECRET']
PINGONE_AUDIENCE = os.environ['INBOUND_PINGONE_AUDIENCE']
PINGONE_SCOPE = os.environ['INBOUND_PINGONE_SCOPE']
PINGONE_TOKEN_URL = os.environ['INBOUND_PINGONE_TOKEN_URL']
PINGONE_DEVICE_AUTH_URL = os.environ['INBOUND_PINGONE_DEVICE_AUTH_URL']
PINGONE_DISCOVERY_URL = os.environ['INBOUND_PINGONE_DISCOVERY_URL']

### 2.3: Discover AWS Environment Identity

In [ ]:
import boto3
from boto3.session import Session

# Create an AWS session from environment variables or ~/.aws/credentials.
boto_session = Session()
region = boto_session.region_name

# Retrieve account ID for verification.
sts = boto3.client('sts')
account_id = sts.get_caller_identity().get('Account')

print(f'📍 Deploying to AWS Account ID {account_id} in {region}.')

### 2.4: Configure Resource Names

Define names for the AWS resources that will be created. Customize these as needed.

In [ ]:
# Agent: The AgentCore Runtime that handles user requests, secured by PingOne JWT auth.
AGENT_NAME = 'pingone-inbound-auth-agent'

# API: Base URL pattern for invoking the deployed agent.
AGENTCORE_API_BASE = f'https://bedrock-agentcore.{region}.amazonaws.com'

# Session: Prefix for generated session IDs.
SESSION_ID_PREFIX = 'pingone-inbound-session'

## Step 3: Deploy an Authenticated AgentCore Runtime

### 3.1: Define Agent Logic
Write the agent entrypoint that AgentCore Runtime will invoke. The handler receives user prompts and session IDs via the payload.

In [ ]:
%%writefile simple_agent.py
from strands import Agent
from bedrock_agentcore.runtime import BedrockAgentCoreApp

app = BedrockAgentCoreApp()
agent = Agent()

@app.entrypoint
def invoke(payload: dict) -> dict:
    """Handle incoming requests from authenticated users."""
    user_message = payload.get('prompt', 'Hello!')
    session_id = payload.get('session_id', 'no-session')

    # Include session context in the prompt for conversation continuity.
    prompt = f'User (Session {session_id}) asks: {user_message}'
    response = agent(prompt)

    return {'result': response.message}

if __name__ == '__main__':
    app.run()

### 3.2: Configure AgentCore Runtime with PingOne Authentication

Set up the agent deployment with a Custom JWT Authorizer that validates tokens against your PingOne environment.

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime

# Create a Runtime instance to manage agent deployment and infrastructure.
agentcore_runtime = Runtime()

# Configure the agent with PingOne JWT authentication.
# The authorizer validates incoming tokens against PingOne before allowing invocation.
try:
    response = agentcore_runtime.configure(
        entrypoint='simple_agent.py',
        auto_create_execution_role=True,
        auto_create_ecr=True,
        requirements_file='../requirements.txt',
        region=region,
        agent_name=AGENT_NAME,
        authorizer_configuration={
            'customJWTAuthorizer': {
                'discoveryUrl': PINGONE_DISCOVERY_URL,
                'allowedClients': [PINGONE_CLIENT_ID],
                'allowedAudience': [PINGONE_AUDIENCE],
                'allowedScopes': PINGONE_SCOPE.split()
            }
        }
    )
    print('✅ Configuration successful')
except Exception as e:
    print(f'❌ Configuration failed: {e}')

### 3.3: Deploy the Agent to AgentCore Runtime

Package and deploy the agent code to AWS. The toolkit builds a container, pushes it to ECR, and provisions the AgentCore Runtime endpoint.

In [ ]:
# Deploy the agent to AWS (builds container, pushes to ECR, provisions endpoint).
launch_result = agentcore_runtime.launch(auto_update_on_conflict=True)

### 3.4: Monitor Deployment Status

Wait for the AgentCore Runtime to reach READY status before proceeding.

In [ ]:
import time

# Poll deployment status until the agent reaches a terminal state.
status_response = agentcore_runtime.status()
status = status_response.endpoint.get('status') if status_response.endpoint else None

while status not in ['READY', 'CREATE_FAILED', 'DELETE_FAILED', 'UPDATE_FAILED']:
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint.get('status') if status_response.endpoint else None
    print(f'Status: {status}')

print(f'✅ Final status: {status}')

### 3.5: Retrieve Agent Deployment Details

Extract the Agent ARN and other deployment metadata needed to invoke the agent.

In [ ]:
# Extract deployment details from the launch result.
if hasattr(launch_result, 'agent_arn') and launch_result.agent_arn:
    AGENT_ARN = launch_result.agent_arn
    AGENT_ID = launch_result.agent_id
    print(f'📝 Agent ARN: {AGENT_ARN}')
    print(f'📝 Agent ID: {AGENT_ID}')
    print(f'📝 ECR URI: {launch_result.ecr_uri}')
else:
    raise RuntimeError('Could not extract Agent ARN from launch result')

# Verify deployment status.
if status == 'READY':
    print('✅ Agent deployed successfully and ready for testing!')
elif status in ['CREATE_FAILED', 'DELETE_FAILED', 'UPDATE_FAILED']:
    raise RuntimeError(f'Agent deployment failed with status: {status}')
else:
    print(f'⚠️ Unexpected status: {status}')

## Step 4: Prepare for Agent Invocation

### 4.1: Define the PingOne Authentication Handler

Use the device authorization grant to authenticate users in this notebook environment.

In [ ]:
import requests
import webbrowser
from requests.auth import HTTPBasicAuth

def get_pingone_token() -> str:
    """Authenticate with PingOne using the device authorization flow."""
    # Initialize device authorization flow.
    auth_response = requests.post(
        PINGONE_DEVICE_AUTH_URL,
        data={'client_id': PINGONE_CLIENT_ID, 'scope': PINGONE_SCOPE},
        auth=HTTPBasicAuth(PINGONE_CLIENT_ID, PINGONE_CLIENT_SECRET)
    )
    auth_response.raise_for_status()
    auth_data = auth_response.json()

    print(f'\n👉 ACTION REQUIRED')
    print(f'1. Go to: {auth_data["verification_uri"]}')
    print(f'2. Enter Code: {auth_data["user_code"]}')
    webbrowser.open(auth_data['verification_uri'])

    # Poll for the access token.
    print('\n⏳ Waiting for you to log in...')
    poll_interval = auth_data.get('interval', 5)
    while True:
        token_response = requests.post(
            PINGONE_TOKEN_URL,
            data={
                'grant_type': 'urn:ietf:params:oauth:grant-type:device_code',
                'device_code': auth_data['device_code'],
                'client_id': PINGONE_CLIENT_ID
            },
            auth=HTTPBasicAuth(PINGONE_CLIENT_ID, PINGONE_CLIENT_SECRET)
        )
        token_data = token_response.json()

        if 'access_token' in token_data:
            print('✅ Successfully authenticated!')
            return token_data['access_token']

        error = token_data.get('error')
        if error == 'authorization_pending':
            time.sleep(poll_interval)
        elif error == 'slow_down':
            poll_interval += 2
            time.sleep(poll_interval)
        else:
            raise RuntimeError(f'Authentication failed: {token_data.get("error_description", error)}')

### 4.2: Define the Agent Invocation Function

Create a helper function to call the agent with a bearer token.

In [ ]:
import urllib.parse
import uuid

def invoke_agent(access_token: str, query: str, session_id: str|None = None) -> tuple:
    """
    Invoke the agent with the given access token and query.

    Args:
        access_token: PingOne JWT access token
        query: The user's message to send to the agent
        session_id: Optional session ID for conversation continuity

    Returns:
        Tuple of (response_dict, session_id)
    """
    escaped_arn = urllib.parse.quote(AGENT_ARN, safe='')
    url = f'{AGENTCORE_API_BASE}/runtimes/{escaped_arn}/invocations?qualifier=DEFAULT'

    # Generate session ID if not provided.
    if not session_id:
        session_id = f'{SESSION_ID_PREFIX}-{int(time.time())}-{uuid.uuid4().hex[:8]}'

    print(f'🚀 Invoking agent with query: {query}')
    print(f'📋 Session ID: {session_id}')

    response = requests.post(
        url,
        headers={
            'Authorization': f'Bearer {access_token}',
            'Content-Type': 'application/json',
            'X-Amzn-Bedrock-AgentCore-Runtime-Session-Id': session_id
        },
        json={'prompt': query, 'session_id': session_id},
        timeout=300
    )
    response.raise_for_status()

    print('✅ Agent response received')
    return response.json(), session_id

## Step 5: Test the Agent with and without Authentication

### 5.1: Test Unauthenticated Request (Should Fail)

First, let's verify that the agent properly rejects unauthenticated requests.

In [ ]:
print('=' * 60)
print('TEST 1: Unauthenticated Request (Should Fail)')
print('=' * 60)

escaped_arn = urllib.parse.quote(AGENT_ARN, safe='')
url = f'{AGENTCORE_API_BASE}/runtimes/{escaped_arn}/invocations?qualifier=DEFAULT'

try:
    response = requests.post(
        url,
        headers={'Content-Type': 'application/json'},
        json={'prompt': 'Hello without authentication'}
    )
    response.raise_for_status()
    print(f'❌ Unexpected success: {response.status_code}')
except requests.exceptions.HTTPError as e:
    print(f'✅ Expected failure: {e.response.status_code}')
    print(f'   Error: {e.response.text}')

### 5.2: Test Authenticated Request

Now let's get a bearer token and test that the agent responds to authenticated requests.

In [ ]:
import json

print('=' * 60)
print('TEST 2: Authenticated Request')
print('=' * 60)

# Get OAuth token from PingOne.
access_token = get_pingone_token()

# Invoke the agent with the authenticated token.
result1, session_id1 = invoke_agent(
    access_token,
    'Hello, can you guide me about AWS security best practices for authentication?'
)
print(f'\nSession ID: {session_id1}')
print(json.dumps(result1, indent=2))

### 5.3: Test Session Continuity

Verify the agent can hold a conversation by reusing the same session ID.

In [ ]:
print('=' * 60)
print('TEST 3: Session Continuity')
print('=' * 60)

# Continue the conversation using the same session ID.
result2, session_id2 = invoke_agent(
    access_token,
    'What was my previous question?',
    session_id=session_id1
)
print(f'\nSession ID: {session_id2}')
print(json.dumps(result2, indent=2))

## Conclusion and Cleanup

### Resource Cleanup

Clean up the resources created in this tutorial.

In [ ]:
# Delete the AgentCore Runtime and clean up the cached configuration.
agentcore_runtime.destroy()

### Conclusion

This notebook demonstrated how to:
1. **Setup PingOne** - Configure a PingOne environment, resource, application, and test user
2. **Configure AWS Environment** - Set up notebook dependencies and AWS credentials
3. **Deploy Authenticated Agent** - Create and deploy an agent with PingOne JWT validation
4. **Prepare Invocation Helpers** - Define authentication and agent invocation functions
5. **Test Authentication Flow** - Verify that unauthenticated requests fail and authenticated requests succeed

### Key Learnings

- **Inbound Authentication:** PingOne protects agent endpoints, ensuring only authenticated users can invoke agents
- **Session Management:** Agents can access session information for personalized responses
- **JWT Token Validation:** AgentCore automatically validates PingOne JWT tokens
- **Security by Default:** Unauthenticated requests are automatically rejected

### Next Steps

- Implement user-based authentication flows
- Add more sophisticated agent logic with user context
- Explore outbound authentication for accessing external APIs
- Integrate with AgentCore Gateway for additional security layers